In [2]:
# IMPORT ALL REQUIRED LIBRARIES (Cell 1 - Must run first!)
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
DATA_DIR = Path(r"c:\Users\WINDOWS\OneDrive\Desktop\Sys\data")
NOTEBOOKS_DIR = Path(r"c:\Users\WINDOWS\OneDrive\Desktop\Sys\notebooks")
INPUT_CSV = DATA_DIR / "CATNAT_2023_2025.csv"
RPA_LOOKUP = DATA_DIR / "rpa_zoning.csv"
OUTPUT_PARQUET = DATA_DIR / "portfolio_enriched.parquet"

print("✓ All libraries imported successfully")
print("=" * 80)

# 1. DATA LOADING & INITIAL EXPLORATION
print("LOADING CATNAT DATASET...")
print("=" * 80)

df = pd.read_csv(INPUT_CSV, encoding='utf-8', decimal=',')
print(f"\n✓ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"✓ Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display basic info
print("\n📊 DATASET OVERVIEW:")
print(df.head(3))
print("\n📋 DATA TYPES & MISSING VALUES:")
print(df.info())
print("\n📈 SUMMARY STATISTICS (Numerical Columns):")
print(df.describe())

✓ All libraries imported successfully
LOADING CATNAT DATASET...

✓ Dataset loaded: 39,196 rows × 10 columns
✓ Memory usage: 13.09 MB

📊 DATASET OVERVIEW:
    NUMERO_POLICE  CODE_SOUS_BRANCHE  NUM_AVNT_COURS  DATE_EFFET  \
0  16330013230012               3302               0  03/01/2023   
1  34303003230006               3302               0  04/01/2023   
2  19306003230396               3302               0  10/05/2023   

  DATE_EXPIRATION                           TYPE             WILAYA  \
0      02/01/2024  1 - Installation Industrielle          9 - BLIDA   
1      03/01/2024  1 - Installation Industrielle  34 - B.B ARRERIDJ   
2      09/05/2024  1 - Installation Industrielle         19 - SETIF   

                    COMMUNE  CAPITAL_ASSURE  PRIME_NETTE  
0            111 - BIRTOUTA       1000000.0       2500.0  
1  368 - BORDJ BOU ARRERIDJ       1000000.0       2500.0  
2           1158 - EL EULMA      60000000.0      25800.0  

📋 DATA TYPES & MISSING VALUES:
<class 'pandas.core.

In [3]:


# 2. DATA CLEANING & STANDARDIZATION

print("\n" + "=" * 80)
print("PHASE 0: DATA CLEANING & STANDARDIZATION")
print("=" * 80)

# Create working copy
df_clean = df.copy()

# === ADD THIS RIGHT AFTER creating df_clean ===
print("🔄 Converting dates to datetime...")
date_cols = ['DATE_EFFET', 'DATE_EXPIRATION']
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], format='%d/%m/%Y', errors='coerce')

print(f"  ✓ Dates parsed. Missing values: {df_clean[date_cols].isna().sum().to_dict()}")

# 2.1 Handle PRIME_NETTE: Check for negative/zero values
print("\n🔍 PRIME_NETTE (Net Premium) Analysis:")
print(f"  - Min: ${df_clean['PRIME_NETTE'].min():,.2f}")
print(f"  - Max: ${df_clean['PRIME_NETTE'].max():,.2f}")
print(f"  - Negative values: {(df_clean['PRIME_NETTE'] < 0).sum()}")
print(f"  - Zero values: {(df_clean['PRIME_NETTE'] == 0).sum()}")

# Flag negative/zero premiums for review
df_clean['PREMIUM_FLAG'] = 'VALID'
df_clean.loc[df_clean['PRIME_NETTE'] <= 0, 'PREMIUM_FLAG'] = 'INVALID'
print(f"  ⚠️  Flagged {(df_clean['PREMIUM_FLAG'] == 'INVALID').sum()} records with invalid premiums")

# 2.2 Handle duplicates
print("\n🔍 DUPLICATE RECORDS:")
exact_dupes = df_clean.duplicated(subset=None, keep=False).sum()
print(f"  - Exact duplicates: {exact_dupes}")
policy_dupes = df_clean.duplicated(subset=['NUMERO_POLICE'], keep=False).sum()
print(f"  - Policy number duplicates (amendments/versions): {policy_dupes}")
print("  ℹ️  Note: Duplicates retained (represent policy versions/amendments)")

# 2.3 Extract WILAYA_CODE and COMMUNE_CODE
print("\n🔍 GEOGRAPHIC FIELD EXTRACTION:")

def extract_code(value):
    """Extract numeric code from strings like '9 - BLIDA' → 9"""
    if pd.isna(value):
        return None
    try:
        return int(str(value).split(' - ')[0].strip())
    except:
        return None

df_clean['WILAYA_CODE'] = df_clean['WILAYA'].apply(extract_code)
df_clean['COMMUNE_CODE'] = df_clean['COMMUNE'].apply(extract_code)

print(f"  ✓ WILAYA_CODE extracted: {df_clean['WILAYA_CODE'].notna().sum():,} records")
print(f"  ✓ COMMUNE_CODE extracted: {df_clean['COMMUNE_CODE'].notna().sum():,} records")
print(f"  ⚠️  Missing WILAYA_CODE: {df_clean['WILAYA_CODE'].isna().sum()}")
print(f"  ⚠️  Missing COMMUNE_CODE: {df_clean['COMMUNE_CODE'].isna().sum()}")

# 2.4 Create POLICY_STATUS
print("\n🔍 POLICY STATUS CREATION:")
df_clean['POLICY_STATUS'] = 'ACTIVE'
df_clean.loc[df_clean['DATE_EXPIRATION'] < pd.Timestamp.now(), 'POLICY_STATUS'] = 'EXPIRED'
df_clean.loc[df_clean['PREMIUM_FLAG'] == 'INVALID', 'POLICY_STATUS'] = 'FLAGGED'

status_counts = df_clean['POLICY_STATUS'].value_counts()
print(f"  - ACTIVE: {status_counts.get('ACTIVE', 0):,}")
print(f"  - EXPIRED: {status_counts.get('EXPIRED', 0):,}")
print(f"  - FLAGGED: {status_counts.get('FLAGGED', 0):,}")

print(f"\n✅ Cleaning complete: {df_clean.shape[0]:,} records prepared")


PHASE 0: DATA CLEANING & STANDARDIZATION
🔄 Converting dates to datetime...
  ✓ Dates parsed. Missing values: {'DATE_EFFET': 0, 'DATE_EXPIRATION': 0}

🔍 PRIME_NETTE (Net Premium) Analysis:
  - Min: $-1,605,315.75
  - Max: $8,268,112.47
  - Negative values: 459
  - Zero values: 1375
  ⚠️  Flagged 1834 records with invalid premiums

🔍 DUPLICATE RECORDS:
  - Exact duplicates: 34
  - Policy number duplicates (amendments/versions): 3609
  ℹ️  Note: Duplicates retained (represent policy versions/amendments)

🔍 GEOGRAPHIC FIELD EXTRACTION:
  ✓ WILAYA_CODE extracted: 39,143 records
  ✓ COMMUNE_CODE extracted: 39,178 records
  ⚠️  Missing WILAYA_CODE: 53
  ⚠️  Missing COMMUNE_CODE: 18

🔍 POLICY STATUS CREATION:
  - ACTIVE: 0
  - EXPIRED: 37,362
  - FLAGGED: 1,834

✅ Cleaning complete: 39,196 records prepared


In [4]:
# 3. RPA 99/VERSION 2003 ZONING INTEGRATION

print("\n" + "=" * 80)
print("RPA 99/VERSION 2003 ZONING MAPPING")
print("=" * 80)

# Load RPA zoning reference table
print("\n📍 Loading RPA Seismic Zoning Reference...")
rpa_df = pd.read_csv(RPA_LOOKUP, encoding='utf-8')
print(f"  ✓ RPA reference: {rpa_df.shape[0]} records")
print(f"  ✓ Unique wilayas: {rpa_df['WILAYA_CODE'].nunique()}")

# Display sample RPA zones
print("\n📊 RPA ZONING SAMPLE (by Zone):")
print(rpa_df.groupby('ZONE_SISMIQUE')[['WILAYA_CODE', 'WILAYA_NAME']].apply(
    lambda x: f"{len(x)} wilayas in zone"
))

# Merge portfolio with RPA zoning
print("\n🔗 Merging Portfolio with RPA Zoning...")
df_enriched = df_clean.merge(
    rpa_df[['WILAYA_CODE', 'ZONE_SISMIQUE', 'RISK_LEVEL']],
    on='WILAYA_CODE',
    how='left'
)

# Check coverage
coverage_pct = (df_enriched['ZONE_SISMIQUE'].notna().sum() / len(df_enriched)) * 100
print(f"  ✓ RPA zone mapping: {coverage_pct:.2f}% coverage ({df_enriched['ZONE_SISMIQUE'].notna().sum():,}/{len(df_enriched):,})")

# Identify unmapped records
unmapped = df_enriched[df_enriched['ZONE_SISMIQUE'].isna()]
if len(unmapped) > 0:
    print(f"  ⚠️  {len(unmapped)} records unmapped - check WILAYA_CODE:")
    print(unmapped[['NUMERO_POLICE', 'WILAYA', 'WILAYA_CODE']].head())
else:
    print("  ✅ 100% COVERAGE ACHIEVED!")

# Zone distribution
print("\n📈 CAPITAL EXPOSED BY SEISMIC ZONE:")
zone_summary = df_enriched.groupby('ZONE_SISMIQUE').agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': 'sum',
    'PRIME_NETTE': 'sum'
}).rename(columns={
    'NUMERO_POLICE': 'Policy_Count',
    'CAPITAL_ASSURE': 'Total_Capital',
    'PRIME_NETTE': 'Total_Premium'
})
zone_summary['Avg_Capital'] = zone_summary['Total_Capital'] / zone_summary['Policy_Count']
zone_summary['Pct_Capital'] = (zone_summary['Total_Capital'] / zone_summary['Total_Capital'].sum() * 100).round(2)

print(zone_summary.to_string())

print(f"\n✅ {len(df_enriched):,} records enriched with ZONE_SISMIQUE and RISK_LEVEL")


RPA 99/VERSION 2003 ZONING MAPPING

📍 Loading RPA Seismic Zoning Reference...
  ✓ RPA reference: 51 records
  ✓ Unique wilayas: 51

📊 RPA ZONING SAMPLE (by Zone):
ZONE_SISMIQUE
0       9 wilayas in zone
I       9 wilayas in zone
III     4 wilayas in zone
IIa    18 wilayas in zone
IIb    11 wilayas in zone
dtype: object

🔗 Merging Portfolio with RPA Zoning...
  ✓ RPA zone mapping: 99.82% coverage (39,126/39,196)
  ⚠️  70 records unmapped - check WILAYA_CODE:
       NUMERO_POLICE WILAYA  WILAYA_CODE
493   19212003230011    NaN          NaN
696   25308003230168    NaN          NaN
1574  42303003230009    NaN          NaN
3344  42304003230221    NaN          NaN
4614  16336003230184    NaN          NaN

📈 CAPITAL EXPOSED BY SEISMIC ZONE:
               Policy_Count  Total_Capital  Total_Premium   Avg_Capital  Pct_Capital
ZONE_SISMIQUE                                                                       
0                      2608   2.318469e+10   8.999412e+06  8.889835e+06         7.75


In [5]:
# 4. VULNERABILITY SCORING & CLASSIFICATION

print("\n" + "=" * 80)
print("VULNERABILITY SCORING (RPA CHAPTER IX RULES)")
print("=" * 80)

# Define vulnerability mapping based on TYPE and ZONE_SISMIQUE
# RPA Chapter IX: Confined masonry vulnerability varies by zone and construction type

def calculate_vulnerability_score(row):
    """
    Calculate vulnerability score (0-1) based on:
    - TYPE: Industrial > Commercial > Real Estate (baseline)
    - ZONE_SISMIQUE: Higher zone = higher vulnerability multiplier
    RPA Chapter IX rules: Confined masonry limits vary by zone
    """
    
    base_scores = {
        '1 - Installation Industrielle': 0.8,      # Industrial: highest vulnerability
        '2 - Installation Commerciale': 0.6,       # Commercial: medium
        '3 - Bien immobilier': 0.4,                # Real Estate: baseline
    }
    
    zone_multipliers = {
        0: 0.2,       # Zone 0: minimal seismic activity
        'I': 0.4,     # Zone I: low
        'IIa': 0.6,   # Zone IIa: medium
        'IIb': 0.8,   # Zone IIb: high
        'III': 1.0    # Zone III: very high (max vulnerability)
    }
    
    type_key = row['TYPE']
    base_score = base_scores.get(type_key, 0.5)
    
    zone = row['ZONE_SISMIQUE']
    zone_mult = zone_multipliers.get(zone, 0.5)
    
    # Combined vulnerability = base × zone modifier
    return min(base_score * zone_mult, 1.0)

print("\n🔬 VULNERABILITY SCORING MODEL:")
print("  Base Score by TYPE:")
for k, v in {'Industrial': 0.8, 'Commercial': 0.6, 'Real Estate': 0.4}.items():
    print(f"    - {k}: {v}")
print("  Zone Multiplier (RPA Chapter IX):")
for k, v in {0: 0.2, 'I': 0.4, 'IIa': 0.6, 'IIb': 0.8, 'III': 1.0}.items():
    print(f"    - Zone {k}: {v}x")

df_enriched['VULNERABILITY_SCORE'] = df_enriched.apply(calculate_vulnerability_score, axis=1)

# Vulnerability distribution
print("\n📊 VULNERABILITY SCORE DISTRIBUTION:")
print(df_enriched['VULNERABILITY_SCORE'].describe().to_string())

print("\n📈 BY TYPE:")
type_vuln = df_enriched.groupby('TYPE')['VULNERABILITY_SCORE'].agg(['mean', 'min', 'max', 'std'])
print(type_vuln.to_string())

print("\n📈 BY SEISMIC ZONE:")
zone_vuln = df_enriched.groupby('ZONE_SISMIQUE')['VULNERABILITY_SCORE'].agg(['mean', 'min', 'max', 'count'])
print(zone_vuln.to_string())

# High vulnerability policies (Zone III + Industrial)
high_vuln = df_enriched[
    (df_enriched['ZONE_SISMIQUE'] == 'III') & 
    (df_enriched['TYPE'] == '1 - Installation Industrielle')
]
print(f"\n🚨 HIGH VULNERABILITY POLICIES (Zone III Industrial):")
print(f"  - Count: {len(high_vuln):,} policies")
print(f"  - Capital exposed: ${high_vuln['CAPITAL_ASSURE'].sum():,.0f}")
print(f"  - Avg score: {high_vuln['VULNERABILITY_SCORE'].mean():.3f}")

print(f"\n✅ Vulnerability scores assigned to all {len(df_enriched):,} records")


VULNERABILITY SCORING (RPA CHAPTER IX RULES)

🔬 VULNERABILITY SCORING MODEL:
  Base Score by TYPE:
    - Industrial: 0.8
    - Commercial: 0.6
    - Real Estate: 0.4
  Zone Multiplier (RPA Chapter IX):
    - Zone 0: 0.2x
    - Zone I: 0.4x
    - Zone IIa: 0.6x
    - Zone IIb: 0.8x
    - Zone III: 1.0x

📊 VULNERABILITY SCORE DISTRIBUTION:
count    39196.000000
mean         0.345844
std          0.087178
min          0.200000
25%          0.300000
50%          0.300000
75%          0.400000
max          0.800000

📈 BY TYPE:
                                   mean   min  max       std
TYPE                                                        
1 - Installation Industrielle  0.570641  0.32  0.8  0.157144
2 - Installation Commerciale   0.385811  0.24  0.6  0.084799
Bien immobilier                0.329304  0.20  0.5  0.077163

📈 BY SEISMIC ZONE:
                   mean   min   max  count
ZONE_SISMIQUE                             
0              0.270130  0.25  0.40   2608
I              0.

In [6]:
# 5. PORTFOLIO SEGMENTATION & AGGREGATION

print("\n" + "=" * 80)
print("PORTFOLIO SEGMENTATION & AGGREGATION")
print("=" * 80)

# Standardize TYPE names
print("\n🏷️  TYPE DISTRIBUTION:")
type_counts = df_enriched['TYPE'].value_counts()
print(type_counts.to_string())

# Segmentation 1: By TYPE
print("\n📊 CAPITAL BY TYPE:")
type_seg = df_enriched.groupby('TYPE').agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': ['sum', 'mean'],
    'PRIME_NETTE': ['sum', 'mean']
})
type_seg.columns = ['Policy_Count', 'Total_Capital', 'Avg_Capital', 'Total_Premium', 'Avg_Premium']
type_seg['Pct_Capital'] = (type_seg['Total_Capital'] / type_seg['Total_Capital'].sum() * 100).round(2)
print(type_seg.to_string())

# Segmentation 2: By ZONE_SISMIQUE
print("\n📊 CAPITAL BY SEISMIC ZONE (Already calculated above)")

# Segmentation 3: By WILAYA
print("\n📊 TOP 15 WILAYAS BY CAPITAL EXPOSED:")
wilaya_seg = df_enriched.groupby(['WILAYA_CODE', 'WILAYA']).agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': 'sum',
    'PRIME_NETTE': 'sum',
    'VULNERABILITY_SCORE': 'mean'
}).sort_values('CAPITAL_ASSURE', ascending=False).head(15)
wilaya_seg.columns = ['Policy_Count', 'Total_Capital', 'Total_Premium', 'Avg_Vulnerability']
print(wilaya_seg.to_string())

# Segmentation 4: By ZONE + TYPE
print("\n📊 CAPITAL BY ZONE × TYPE (Crosstab):")
pivot = df_enriched.pivot_table(
    values='CAPITAL_ASSURE',
    index='ZONE_SISMIQUE',
    columns='TYPE',
    aggfunc='sum',
    margins=True
)
print(pivot.to_string())

# Segmentation 5: By WILAYA + ZONE
print("\n📊 TOP 20 WILAYA × ZONE COMBINATIONS:")
wilaya_zone = df_enriched.groupby(['WILAYA_CODE', 'WILAYA', 'ZONE_SISMIQUE']).agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': 'sum',
    'VULNERABILITY_SCORE': 'mean'
}).sort_values('CAPITAL_ASSURE', ascending=False).head(20)
wilaya_zone.columns = ['Policy_Count', 'Total_Capital', 'Avg_Vulnerability']
print(wilaya_zone.to_string())

print(f"\n✅ Portfolio segmentation complete")


PORTFOLIO SEGMENTATION & AGGREGATION

🏷️  TYPE DISTRIBUTION:
TYPE
Bien immobilier                  29100
2 - Installation Commerciale      9675
1 - Installation Industrielle      421

📊 CAPITAL BY TYPE:
                               Policy_Count  Total_Capital   Avg_Capital  Total_Premium   Avg_Premium  Pct_Capital
TYPE                                                                                                              
1 - Installation Industrielle           421   9.921191e+10  2.356577e+08   3.171837e+07  75340.558162        33.12
2 - Installation Commerciale           9675   8.992777e+10  9.295821e+06   3.981196e+07   4114.931499        30.02
Bien immobilier                       29100   1.104025e+11  3.793901e+06   7.489610e+07   2573.749127        36.86

📊 CAPITAL BY SEISMIC ZONE (Already calculated above)

📊 TOP 15 WILAYAS BY CAPITAL EXPOSED:
                               Policy_Count  Total_Capital  Total_Premium  Avg_Vulnerability
WILAYA_CODE WILAYA                  

In [7]:
# 6. CONCENTRATION ANALYSIS & HOTSPOT IDENTIFICATION

print("\n" + "=" * 80)
print("CONCENTRATION ANALYSIS & HOTSPOT IDENTIFICATION")
print("=" * 80)

# Calculate company-wide retention capacity (use total capital as reference)
total_capital = df_enriched['CAPITAL_ASSURE'].sum()
total_policies = len(df_enriched)
retention_capacity_pct = 2.0  # Define threshold: 2% of total capital in one commune = concentration risk

print(f"\n📊 PORTFOLIO METRICS:")
print(f"  - Total capital exposed: ${total_capital:,.0f}")
print(f"  - Total policies: {total_policies:,}")
print(f"  - Concentration threshold: {retention_capacity_pct}% of total capital")
print(f"  - Dollar threshold: ${total_capital * retention_capacity_pct / 100:,.0f}")

# Concentration by COMMUNE
print("\n🔴 SURCONCENTRATION ANALYSIS (COMMUNE LEVEL):")
commune_conc = df_enriched.groupby(['WILAYA_CODE', 'WILAYA', 'COMMUNE_CODE', 'COMMUNE', 'ZONE_SISMIQUE']).agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': 'sum',
    'PRIME_NETTE': 'sum',
    'VULNERABILITY_SCORE': 'mean'
}).sort_values('CAPITAL_ASSURE', ascending=False)

commune_conc.columns = ['Policy_Count', 'Total_Capital', 'Total_Premium', 'Avg_Vulnerability']
commune_conc['Pct_Total_Capital'] = (commune_conc['Total_Capital'] / total_capital * 100).round(2)
commune_conc['Concentration_Flag'] = commune_conc['Pct_Total_Capital'].apply(
    lambda x: '🚨 CRITICAL' if x >= retention_capacity_pct else ('⚠️  HIGH' if x >= retention_capacity_pct/2 else '✓ OK')
)

# Top hotspots
top_hotspots = commune_conc.head(20)
print("\n📍 TOP 20 SURCONCENTRATION HOTSPOTS:")
print(top_hotspots.to_string())

# Zone III concentration
print("\n🚨 ZONE III SPECIAL FOCUS (Highest Seismic Risk):")
zone_iii = df_enriched[df_enriched['ZONE_SISMIQUE'] == 'III']
zone_iii_capital = zone_iii['CAPITAL_ASSURE'].sum()
zone_iii_pct = (zone_iii_capital / total_capital * 100)
print(f"  - Policies in Zone III: {len(zone_iii):,} ({len(zone_iii)/total_policies*100:.2f}%)")
print(f"  - Capital exposed in Zone III: ${zone_iii_capital:,.0f} ({zone_iii_pct:.2f}% of total)")
print(f"  - ⚠️  {zone_iii_pct:.2f}% is {'EXCESSIVE' if zone_iii_pct > 15 else 'MANAGEABLE' if zone_iii_pct > 10 else 'LOW'}")

zone_iii_communes = zone_iii.groupby('COMMUNE').agg({
    'CAPITAL_ASSURE': 'sum',
    'NUMERO_POLICE': 'count'
}).sort_values('CAPITAL_ASSURE', ascending=False).head(10)
print("\n  Top Zone III Communes by Capital:")
print(zone_iii_communes.to_string())

# Concentration by WILAYA
print("\n🔴 SURCONCENTRATION BY WILAYA:")
wilaya_conc = df_enriched.groupby(['WILAYA_CODE', 'WILAYA', 'ZONE_SISMIQUE']).agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': 'sum'
}).sort_values('CAPITAL_ASSURE', ascending=False)
wilaya_conc.columns = ['Policy_Count', 'Total_Capital']
wilaya_conc['Pct_Total_Capital'] = (wilaya_conc['Total_Capital'] / total_capital * 100).round(2)
print(wilaya_conc.head(15).to_string())

# Export hotspots for Phase I
hotspots_export = top_hotspots.reset_index()[['WILAYA', 'COMMUNE', 'ZONE_SISMIQUE', 'Policy_Count', 'Total_Capital', 'Pct_Total_Capital', 'Concentration_Flag']]
hotspots_export.to_csv(DATA_DIR / 'hotspots_identified.csv', index=False, decimal=',')
print(f"\n✅ Hotspots exported to hotspots_identified.csv")

print(f"\n✅ Concentration analysis complete - {len(commune_conc)} communes analyzed")


CONCENTRATION ANALYSIS & HOTSPOT IDENTIFICATION

📊 PORTFOLIO METRICS:
  - Total capital exposed: $299,542,211,932
  - Total policies: 39,196
  - Concentration threshold: 2.0% of total capital
  - Dollar threshold: $5,990,844,239

🔴 SURCONCENTRATION ANALYSIS (COMMUNE LEVEL):

📍 TOP 20 SURCONCENTRATION HOTSPOTS:
                                                                                   Policy_Count  Total_Capital  Total_Premium  Avg_Vulnerability  Pct_Total_Capital Concentration_Flag
WILAYA_CODE WILAYA            COMMUNE_CODE COMMUNE                  ZONE_SISMIQUE                                                                                                     
16.0        16 - ALGER        99.0         99 - ALGER               IIa                     447   4.080454e+10   1.978086e+07           0.314765              13.62         🚨 CRITICAL
31.0        31 - ORAN         1019.0       1019 - ORAN              IIa                     999   2.782015e+10   6.373864e+06           0.

In [8]:
# 7. DATA EXPORT & DELIVERABLES

print("\n" + "=" * 80)
print("EXPORTING ENRICHED DATASETS")
print("=" * 80)

# Select columns for output
output_cols = [
    'NUMERO_POLICE', 'CODE_SOUS_BRANCHE', 'NUM_AVNT_COURS',
    'DATE_EFFET', 'DATE_EXPIRATION', 'TYPE',
    'WILAYA', 'COMMUNE', 'WILAYA_CODE', 'COMMUNE_CODE',
    'CAPITAL_ASSURE', 'PRIME_NETTE',
    'ZONE_SISMIQUE', 'RISK_LEVEL', 'VULNERABILITY_SCORE',
    'POLICY_STATUS', 'PREMIUM_FLAG'
]

df_output = df_enriched[output_cols].copy()

# Export 1: Parquet (main enriched dataset)
df_output.to_parquet(OUTPUT_PARQUET, index=False, compression='snappy')
print(f"\n✓ Portfolio enriched: {OUTPUT_PARQUET}")
print(f"  - Records: {len(df_output):,}")
print(f"  - Size: {os.path.getsize(OUTPUT_PARQUET) / 1024**2:.2f} MB")

# Export 2: CSV format (alternative)
csv_path = DATA_DIR / 'portfolio_enriched.csv'
df_output.to_csv(csv_path, index=False, decimal=',')
print(f"\n✓ CSV export: {csv_path}")

# Export 3: Data quality report
quality_report = {
    'Total_Records': len(df_output),
    'Active_Policies': (df_output['POLICY_STATUS'] == 'ACTIVE').sum(),
    'Expired_Policies': (df_output['POLICY_STATUS'] == 'EXPIRED').sum(),
    'Flagged_Policies': (df_output['POLICY_STATUS'] == 'FLAGGED').sum(),
    'RPA_Coverage_Pct': (df_output['ZONE_SISMIQUE'].notna().sum() / len(df_output) * 100),
    'Total_Capital': df_output['CAPITAL_ASSURE'].sum(),
    'Total_Premium': df_output['PRIME_NETTE'].sum(),
    'Unique_Wilayas': df_output['WILAYA_CODE'].nunique(),
    'Unique_Communes': df_output['COMMUNE_CODE'].nunique(),
    'Policy_Duplicates': (df_enriched.duplicated(subset=['NUMERO_POLICE'], keep=False).sum()),
}

quality_df = pd.DataFrame([quality_report]).T
quality_df.columns = ['Value']
quality_df.to_csv(DATA_DIR / 'phase0_quality_metrics.csv')
print(f"\n✓ Quality metrics: {DATA_DIR / 'phase0_quality_metrics.csv'}")

print("\n" + "=" * 80)
print("PHASE 0 COMPLETION SUMMARY")
print("=" * 80)

print(f"""
✅ DATA PREPARATION COMPLETED

📊 DATASET STATISTICS:
   • Total Records: {len(df_output):,}
   • Active Policies: {(df_output['POLICY_STATUS'] == 'ACTIVE').sum():,}
   • Total Capital: ${df_output['CAPITAL_ASSURE'].sum():,.0f}
   • Total Premium: ${df_output['PRIME_NETTE'].sum():,.0f}

🗺️  GEOGRAPHIC COVERAGE:
   • Wilayas: {df_output['WILAYA_CODE'].nunique()} / 51 ({df_output['WILAYA_CODE'].nunique()/51*100:.1f}%)
   • Communes: {df_output['COMMUNE_CODE'].nunique()} / 789 ({df_output['COMMUNE_CODE'].nunique()/789*100:.1f}%)
   • RPA Zone Mapping: {(df_output['ZONE_SISMIQUE'].notna().sum() / len(df_output) * 100):.2f}%

🎯 RPA SEISMIC ZONES:
   • Zone 0 (Very Low): {(df_output['ZONE_SISMIQUE'] == 0).sum():,} policies
   • Zone I (Low): {(df_output['ZONE_SISMIQUE'] == 'I').sum():,} policies
   • Zone IIa (Medium-High): {(df_output['ZONE_SISMIQUE'] == 'IIa').sum():,} policies
   • Zone IIb (High): {(df_output['ZONE_SISMIQUE'] == 'IIb').sum():,} policies
   • Zone III (Very High): {(df_output['ZONE_SISMIQUE'] == 'III').sum():,} policies

📁 OUTPUTS CREATED:
   ✓ portfolio_enriched.parquet
   ✓ portfolio_enriched.csv
   ✓ hotspots_identified.csv
   ✓ phase0_quality_metrics.csv

🚀 READY FOR PHASE I: Risk Identification & Classification
""")

print("\nNext Steps:")
print("  1. Review hotspots_identified.csv for surconcentration patterns")
print("  2. Validate ZONE_SISMIQUE mapping against official RPA Annexe 1")
print("  3. Schedule Phase I kickoff: Vulnerability Scoring & PML Analysis")


EXPORTING ENRICHED DATASETS

✓ Portfolio enriched: c:\Users\WINDOWS\OneDrive\Desktop\Sys\data\portfolio_enriched.parquet
  - Records: 39,196
  - Size: 0.74 MB

✓ CSV export: c:\Users\WINDOWS\OneDrive\Desktop\Sys\data\portfolio_enriched.csv

✓ Quality metrics: c:\Users\WINDOWS\OneDrive\Desktop\Sys\data\phase0_quality_metrics.csv

PHASE 0 COMPLETION SUMMARY

✅ DATA PREPARATION COMPLETED

📊 DATASET STATISTICS:
   • Total Records: 39,196
   • Active Policies: 0
   • Total Capital: $299,542,211,932
   • Total Premium: $146,426,437

🗺️  GEOGRAPHIC COVERAGE:
   • Wilayas: 51 / 51 (100.0%)
   • Communes: 789 / 789 (100.0%)
   • RPA Zone Mapping: 99.82%

🎯 RPA SEISMIC ZONES:
   • Zone 0 (Very Low): 0 policies
   • Zone I (Low): 3,258 policies
   • Zone IIa (Medium-High): 20,952 policies
   • Zone IIb (High): 9,172 policies
   • Zone III (Very High): 3,136 policies

📁 OUTPUTS CREATED:
   ✓ portfolio_enriched.parquet
   ✓ portfolio_enriched.csv
   ✓ hotspots_identified.csv
   ✓ phase0_quality_me